# 🧊 Script 1 — Multi-Image SfM 3D Reconstruction
### COLMAP → MiDaS Depth → Open3D → ICP → Tesla Coloring → GLB Export

---

## 📋 How to Use This Notebook

Follow these steps **in order**. Each section is one cell. Run them top to bottom.

### Before you start:
1. **Upload your images to Google Drive** inside a folder (e.g. `MyDrive/3D_Project/images/`)
2. **Use CPU runtime**: this notebook is configured for CPU stability by default.
3. Run **Cell 1** to mount Drive, then **Cell 2** to install tools (takes ~3 minutes)
4. **Edit Cell 3** (Configuration) — this is the only cell you need to edit
5. Run all remaining cells top to bottom
6. Your `.glb` file will download automatically at the end

### Pipeline stages:
| Cell | Stage | What happens |
|------|-------|--------------|
| 1 | Mount Drive | Connect your Google Drive |
| 2 | Install | COLMAP, Open3D, trimesh, MiDaS |
| 3 | ⚙️ Config | **You edit this cell** — set your paths + settings |
| 4 | Imports | Load all Python libraries |
| 5 | COLMAP SfM | Feature extraction → matching → sparse reconstruction |
| 6 | Parse COLMAP | Read camera poses + sparse 3D points |
| 7 | MiDaS Depth | Estimate depth map per image using AI |
| 8 | Point Clouds | Back-project depth → 3D points per frame |
| 9 | ICP Align | Align frames to fix depth-scale mismatches |
| 10 | Clean | Remove noise (Statistical Outlier Removal) |
| 11 | Tesla Color | Apply plasma/height colormap (Tesla BEV style) |
| 12 | Mesh | Poisson surface reconstruction |
| 13 | Voxel Mesh | Tesla occupancy-grid cube mesh |
| 14 | Export | Save .PLY + .GLB, then auto-download |

---
> **View your .GLB online**: drag and drop to [gltf-viewer.donmccurdy.com](https://gltf-viewer.donmccurdy.com)

## Cell 1 — Mount Google Drive
Run this first. A permission popup will appear — click Allow.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')
print('Your files live at: /content/drive/MyDrive/')

## Cell 2 — Install Dependencies
This takes **2–4 minutes** the first time. After that, keep the runtime alive to avoid reinstalling.

> ⚠️ If you restart the runtime, you must re-run this cell.

In [ ]:
import os, subprocess, sys

print('?? Installing COLMAP...')
subprocess.run(['apt-get', 'install', '-y', '-q', 'colmap'], check=True)

print('?? Installing Python packages...')
pkgs = [
    'open3d',
    'trimesh',
    'scipy',
    'matplotlib',
    'Pillow',
    'opencv-python-headless',   # headless = no display needed
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)

# Verify COLMAP
r = subprocess.run(['colmap', '--help'], capture_output=True, text=True)
colmap_ok = 'COLMAP' in (r.stdout + r.stderr)
print(f'\n? COLMAP installed: {colmap_ok}')

# Force CPU-only mode for stability
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import torch
device = 'cpu'
print(f'? PyTorch device forced to: {device}')
if torch.cuda.is_available():
    print('   ?? GPU is available but intentionally disabled for stability (CPU mode).')

print('\n? All dependencies installed.')


## Cell 3 — ⚙️ Configuration  *(Edit This Cell)*

This is the **only cell you need to edit**. Change the values to match your setup.

### Minimum required changes:
- Set `IMAGE_FOLDER` to the path of your images on Google Drive
- Set `APPLICATION_MODE` to match what you are scanning
- Set `OUTPUT_FOLDER` to where results should be saved

Everything else has sensible defaults for a first run.

In [ ]:
# ============================================================
# ⚙️  CONFIGURATION — Edit values below to match your project
# ============================================================

# ── Application Mode ─────────────────────────────────────────
# Choose what you are scanning. This auto-tunes processing settings.
# Options: 'general' | 'architectural' | 'medical' | 'ecommerce'
#          'autonomous' | 'construction'
APPLICATION_MODE = 'general'

# ── Paths ────────────────────────────────────────────────────
# Folder on Google Drive containing your input images (.jpg / .png)
# Example: '/content/drive/MyDrive/3D_Project/images/'
IMAGE_FOLDER  = '/content/drive/MyDrive/3D_Reconstruction_Files&data/data_img'

# Where to save all output files (point clouds, meshes, GLB)
OUTPUT_FOLDER = '/content/drive/MyDrive/3D_Reconstruction_Files&data/output/'

# ── COLMAP Settings ──────────────────────────────────────────
# 'exhaustive' → compare every image pair. Best for objects/rooms < 100 images
# 'sequential' → compare consecutive images. Best for video frames or walkthroughs
COLMAP_MATCHER = 'exhaustive'

# Image quality for feature extraction
# 'low' = fastest | 'medium' = balanced | 'high' = most features
COLMAP_IMAGE_QUALITY = 'high'

# ── Depth Model ──────────────────────────────────────────────
# 'MiDaS_small' → Fastest. Good for first tests.
# 'DPT_Hybrid'  ? Better quality but heavier memory usage.
# 'DPT_Large'   → Best quality, slowest.
DEPTH_MODEL = 'MiDaS_small'  # CPU-safe default

# Max frames to process for dense reconstruction.
# Start small (10-15) for your first run. Increase when satisfied.
MAX_IMAGES_FOR_DENSE = 10
DEPTH_MAX_OUTPUT_RESOLUTION = 1600  # cap depth map size to avoid memory spikes

# ── Point Cloud Settings ─────────────────────────────────────
# Voxel size for downsampling (meters). Smaller = more detail but slower.
# Object scans: 0.005 | Room: 0.02 | Building/outdoor: 0.05-0.1
VOXEL_SIZE = 0.05

# Noise removal settings
SOR_NB_NEIGHBORS = 20   # Number of neighbors to analyze per point
SOR_STD_RATIO    = 2.0  # Points beyond N std deviations are removed

# ── ICP Registration ─────────────────────────────────────────
ICP_MAX_CORRESPONDENCE_DISTANCE = 0.05
ICP_MAX_ITERATIONS = 50

# ── Tesla-Style Coloring ─────────────────────────────────────
# 'height'   → Color by Z-height using plasma colormap (Tesla BEV style)
# 'rgb'      → Keep original photo colors
# 'distance' → Color by distance from scene center
# 'normals'  → Color by surface orientation
COLOR_MODE = 'height'
COLOR_MAP  = 'plasma'   # 'plasma' | 'turbo' | 'viridis' | 'inferno'

HEIGHT_CLIP_LOW  = 5    # Ignore bottom 5% heights (floor noise)
HEIGHT_CLIP_HIGH = 95   # Ignore top 5% heights (ceiling noise)

# ── Mesh Reconstruction ──────────────────────────────────────
POISSON_DEPTH         = 8     # 8=fast/rough | 9=balanced | 10=fine/slow
MESH_DENSITY_QUANTILE = 0.01  # Remove bottom 1% density artifacts

# ── Voxel Mesh (Tesla BEV Occupancy Style) ───────────────────
GENERATE_VOXEL_MESH   = True
VOXEL_MESH_CUBE_SIZE  = 0.04

# ── Export ───────────────────────────────────────────────────
EXPORT_PLY  = True
EXPORT_GLB  = True
OUTPUT_NAME = 'reconstruction'

# ── Visualization ────────────────────────────────────────────
# Always False in Colab (no display). We use matplotlib instead.
VISUALIZE_INTERMEDIATE = False
VISUALIZE_FINAL        = False

# =============================================================
print('✅ Configuration loaded.')
print(f'   Application mode : {APPLICATION_MODE}')
print(f'   Image folder     : {IMAGE_FOLDER}')
print(f'   Output folder    : {OUTPUT_FOLDER}')
print(f'   COLMAP matcher   : {COLMAP_MATCHER}')
print(f'   Depth model      : {DEPTH_MODEL}')
print(f'   Max dense frames : {MAX_IMAGES_FOR_DENSE}')
print(f'   Voxel size       : {VOXEL_SIZE}m')
print(f'   Color mode       : {COLOR_MODE} ({COLOR_MAP})')

## Cell 4 — Import Libraries
No editing needed. Just run.

In [ ]:
import os, sys, glob, copy, time, subprocess, warnings
import numpy as np
import open3d as o3d
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from scipy.spatial import cKDTree
import torch
import trimesh
warnings.filterwarnings('ignore')

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_FOLDER, 'colmap_sparse'), exist_ok=True)

# Apply application preset overrides
PRESETS = {
    'general'      : {},
    'architectural': {'COLMAP_MATCHER':'sequential','VOXEL_SIZE':0.03,'POISSON_DEPTH':10,'COLOR_MODE':'normals','COLOR_MAP':'viridis'},
    'medical'      : {'VOXEL_SIZE':0.001,'POISSON_DEPTH':10,'COLOR_MAP':'inferno','SOR_STD_RATIO':1.5,'MESH_DENSITY_QUANTILE':0.001},
    'ecommerce'    : {'VOXEL_SIZE':0.005,'POISSON_DEPTH':10,'COLOR_MODE':'rgb'},
    'autonomous'   : {'COLMAP_MATCHER':'sequential','VOXEL_SIZE':0.1,'POISSON_DEPTH':8,'COLOR_MODE':'height','COLOR_MAP':'plasma','VOXEL_MESH_CUBE_SIZE':0.15},
    'construction' : {'COLMAP_MATCHER':'sequential','VOXEL_SIZE':0.08,'POISSON_DEPTH':8,'COLOR_MAP':'turbo','VOXEL_MESH_CUBE_SIZE':0.1},
}
for k, v in PRESETS.get(APPLICATION_MODE, {}).items():
    globals()[k] = v
    print(f'   [PRESET] {k} = {v}')

device = torch.device('cpu')
print(f'\n✅ Libraries loaded. Device: {device}')
print(f'   Open3D version  : {o3d.__version__}')
print(f'   PyTorch version : {torch.__version__}')


## Cell 5 — COLMAP Structure-from-Motion

This cell runs the full COLMAP pipeline:
1. **Feature extraction** — finds SIFT keypoints in every image
2. **Feature matching** — finds which keypoints appear across multiple images
3. **SfM mapper** — reconstructs 3D points + camera poses from matches
4. **Convert to text** — saves output in a readable format

> ⏱️ **Time estimate**: 5–30 minutes depending on number of images and overlap.
> The SfM mapper step is the slowest — watch the output for progress.

In [ ]:
def run_colmap(image_folder, output_folder, matcher, quality):
    db        = os.path.join(output_folder, 'colmap.db')
    sparse    = os.path.join(output_folder, 'colmap_sparse')
    q_map     = {'low':'1000','medium':'2000','high':'3200'}
    max_size  = q_map.get(quality, '2000')
    os.makedirs(sparse, exist_ok=True)

    def run(cmd, label):
        print(f'  ▶ {label}...')
        r = subprocess.run(cmd, capture_output=True, text=True)
        for line in r.stdout.splitlines():
            if any(k in line for k in ['images','point','register','ERROR']):
                print(f'    [colmap] {line}')
        if r.returncode != 0 and 'error' in r.stderr.lower():
            print('  STDERR:', r.stderr[-1000:])
        print(f'  ✅ {label} done.')

    run(['colmap','feature_extractor',
         '--database_path', db,
         '--image_path', image_folder,
         '--SiftExtraction.use_gpu','0',
         '--SiftExtraction.max_image_size', max_size], 'Feature extraction')

    run(['colmap', f'{matcher}_matcher',
         '--database_path', db,
         '--SiftMatching.use_gpu','0'], f'{matcher.capitalize()} matching')

    run(['colmap','mapper',
         '--database_path', db,
         '--image_path', image_folder,
         '--output_path', sparse,
         '--Mapper.min_num_matches','15'], 'SfM mapper (slowest step)')

    recon_dirs = sorted([d for d in os.listdir(sparse)
                         if os.path.isdir(os.path.join(sparse, d))])
    if not recon_dirs:
        raise RuntimeError(
            'COLMAP produced no reconstruction.\n'
            'Common fixes:\n'
            '  • Need at least 10–15 images with 60-70% overlap\n'
            '  • Avoid blurry images and plain textureless surfaces\n'
            '  • Try COLMAP_MATCHER = "sequential" for ordered images'
        )

    bin_path = os.path.join(sparse, recon_dirs[0])
    txt_path = bin_path + '_txt'
    os.makedirs(txt_path, exist_ok=True)

    run(['colmap','model_converter',
         '--input_path', bin_path,
         '--output_path', txt_path,
         '--output_type','TXT'], 'Convert to text')

    return txt_path

# Check images exist
img_files = sorted(glob.glob(os.path.join(IMAGE_FOLDER,'*.jpg')) +
                   glob.glob(os.path.join(IMAGE_FOLDER,'*.jpeg')) +
                   glob.glob(os.path.join(IMAGE_FOLDER,'*.png')))
if not img_files:
    raise FileNotFoundError(f'No images found in: {IMAGE_FOLDER}')
print(f'Found {len(img_files)} images in {IMAGE_FOLDER}')

# Run COLMAP
t0 = time.time()
TXT_FOLDER = run_colmap(IMAGE_FOLDER, OUTPUT_FOLDER, COLMAP_MATCHER, COLMAP_IMAGE_QUALITY)
print(f'\n✅ COLMAP complete in {(time.time()-t0)/60:.1f} min')
print(f'   Output text folder: {TXT_FOLDER}')

## Cell 6 — Parse COLMAP Output
Reads the camera intrinsics, image poses (rotation + translation), and sparse 3D points from COLMAP's text files.

In [ ]:
def quat_to_R(qw, qx, qy, qz):
    n = np.sqrt(qw**2+qx**2+qy**2+qz**2)
    qw,qx,qy,qz = qw/n,qx/n,qy/n,qz/n
    return np.array([
        [1-2*(qy**2+qz**2), 2*(qx*qy-qw*qz), 2*(qx*qz+qw*qy)],
        [2*(qx*qy+qw*qz), 1-2*(qx**2+qz**2), 2*(qy*qz-qw*qx)],
        [2*(qx*qz-qw*qy), 2*(qy*qz+qw*qx), 1-2*(qx**2+qy**2)]])

def parse_cameras(f):
    cams = {}
    for line in open(f):
        if line.startswith('#') or not line.strip(): continue
        p = line.strip().split()
        cid,model,w,h = int(p[0]),p[1],int(p[2]),int(p[3])
        params = [float(x) for x in p[4:]]
        cam = {'model':model,'width':w,'height':h,'dist_coeffs':np.zeros(5)}
        if model=='PINHOLE':            cam['fx'],cam['fy'],cam['cx'],cam['cy']=params[0],params[1],params[2],params[3]
        elif model=='SIMPLE_PINHOLE':   cam['fx']=cam['fy']=params[0]; cam['cx'],cam['cy']=params[1],params[2]
        elif model in('RADIAL','SIMPLE_RADIAL'): cam['fx']=cam['fy']=params[0]; cam['cx'],cam['cy']=params[1],params[2]
        elif model=='OPENCV':           cam['fx'],cam['fy'],cam['cx'],cam['cy']=params[0],params[1],params[2],params[3]
        else:                           cam['fx']=cam['fy']=params[0] if params else float(w); cam['cx']=w/2; cam['cy']=h/2
        cams[cid] = cam
    return cams

def parse_images(f):
    imgs = {}
    lines = [l for l in open(f) if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(lines)-1:
        p = lines[i].strip().split()
        if len(p) < 10: i+=1; continue
        qw,qx,qy,qz = float(p[1]),float(p[2]),float(p[3]),float(p[4])
        tx,ty,tz     = float(p[5]),float(p[6]),float(p[7])
        imgs[p[9]] = {'camera_id':int(p[8]),'R':quat_to_R(qw,qx,qy,qz),'t':np.array([tx,ty,tz])}
        i += 2
    return imgs

def parse_points3d(f):
    pts,cols=[],[]
    for line in open(f):
        if line.startswith('#') or not line.strip(): continue
        p = line.strip().split()
        if len(p)<8: continue
        pts.append([float(p[1]),float(p[2]),float(p[3])])
        cols.append([int(p[4])/255.,int(p[5])/255.,int(p[6])/255.])
    return np.array(pts,dtype=np.float64), np.array(cols,dtype=np.float32)

CAMERAS  = parse_cameras(os.path.join(TXT_FOLDER,'cameras.txt'))
IMAGES   = parse_images(os.path.join(TXT_FOLDER,'images.txt'))
PTS3D, COLS3D = parse_points3d(os.path.join(TXT_FOLDER,'points3D.txt'))

print(f'✅ COLMAP parsed:')
print(f'   Cameras  : {len(CAMERAS)}')
print(f'   Images   : {len(IMAGES)}')
print(f'   3D points: {len(PTS3D):,}')

# Visualize sparse cloud with matplotlib
fig = plt.figure(figsize=(10,5))
ax1 = fig.add_subplot(121,projection='3d')
ax1.scatter(PTS3D[::5,0],PTS3D[::5,1],PTS3D[::5,2],c=COLS3D[::5],s=0.5)
ax1.set_title('COLMAP Sparse Cloud')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax2 = fig.add_subplot(122,projection='3d')
cam_pts = []
for meta in list(IMAGES.values())[:30]:
    pos = -meta['R'].T @ meta['t']
    cam_pts.append(pos)
cam_pts = np.array(cam_pts)
ax2.scatter(cam_pts[:,0],cam_pts[:,1],cam_pts[:,2],c='red',s=20)
ax2.set_title('Camera Positions (red=cameras)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER,'colmap_sparse_cloud.png'),dpi=150,bbox_inches='tight')
plt.show()
print('   Preview saved to output folder.')

## Cell 7 — Load MiDaS Depth Model
Downloads model weights on first run (~100–400 MB). Subsequent runs use the cache.

> CPU mode is enforced by default for stability. Use `MiDaS_small` first, then scale up carefully.

In [ ]:
print(f'Loading MiDaS: {DEPTH_MODEL} on forced {device}...')
MIDAS = torch.hub.load('intel-isl/MiDaS', DEPTH_MODEL, trust_repo=True)
MIDAS.to(device).eval()
midas_transforms = torch.hub.load('intel-isl/MiDaS','transforms',trust_repo=True)
MIDAS_TRANSFORM = (midas_transforms.dpt_transform
                   if DEPTH_MODEL in ('DPT_Large','DPT_Hybrid')
                   else midas_transforms.small_transform)
print(f'✅ MiDaS ready.')

# Test on first image
test_img = cv2.imread(img_files[0])
test_rgb = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)
with torch.no_grad():
    inp = MIDAS_TRANSFORM(test_rgb).to(device)
    pred = MIDAS(inp)
    pred = torch.nn.functional.interpolate(
        pred.unsqueeze(1), size=test_rgb.shape[:2], mode='bicubic', align_corners=False
    ).squeeze().cpu().numpy()
del inp
pred = 1.0/(pred+1e-8)
pred = (pred-pred.min())/(pred.max()-pred.min()+1e-8)

fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].imshow(test_rgb); axes[0].set_title('Original Image'); axes[0].axis('off')
axes[1].imshow(pred, cmap='plasma'); axes[1].set_title('MiDaS Depth (plasma)'); axes[1].axis('off')
plt.suptitle('Depth Estimation Test — First Image', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER,'depth_test.png'),dpi=150,bbox_inches='tight')
plt.show()
print('   Depth test preview saved.')

## Cell 8 — Generate Dense Frame Point Clouds
For each registered image:
1. Run MiDaS → get depth map
2. Scale depth using COLMAP sparse points (metric calibration)
3. Back-project every pixel using the pinhole camera model → 3D point

In [ ]:
def estimate_depth(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h,w = rgb.shape[:2]
    out_h,out_w = h,w
    if isinstance(DEPTH_MAX_OUTPUT_RESOLUTION, int) and DEPTH_MAX_OUTPUT_RESOLUTION > 0:
        max_side = max(h, w)
        if max_side > DEPTH_MAX_OUTPUT_RESOLUTION:
            scale = DEPTH_MAX_OUTPUT_RESOLUTION / float(max_side)
            out_w = max(1, int(round(w * scale)))
            out_h = max(1, int(round(h * scale)))
    with torch.no_grad():
        inp  = MIDAS_TRANSFORM(rgb).to(device)
        pred = MIDAS(inp)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1),size=(out_h,out_w),mode='bicubic',align_corners=False).squeeze()
    d = pred.cpu().numpy().astype(np.float32)
    del pred, inp
    d = 1.0/(d+1e-8)
    d = (d-d.min())/(d.max()-d.min()+1e-8)
    return d

def depth_scale(R, t, sparse_pts):
    if len(sparse_pts)==0: return 1.0
    pts_cam = sparse_pts @ R.T + t
    front   = pts_cam[:,2]>0.01
    if front.sum()<3: return 1.0
    return float(np.clip(np.median(pts_cam[front,2])/0.5, 0.01, 1000.0))

def depth_to_pcd(depth, img_bgr, cam, R, t, scale):
    hd,wd = depth.shape
    hi,wi = img_bgr.shape[:2]
    fx = cam['fx']*(wd/wi); fy = cam['fy']*(hd/hi)
    cx = cam['cx']*(wd/wi); cy = cam['cy']*(hd/hi)
    u,v  = np.meshgrid(np.arange(wd,dtype=np.float32),np.arange(hd,dtype=np.float32))
    dm   = depth * scale
    mask = (dm>0.01)&(dm<50.0)
    x = (u[mask]-cx)*dm[mask]/fx
    y = (v[mask]-cy)*dm[mask]/fy
    z = dm[mask]
    pts = np.stack([x,y,z],axis=-1)
    pts = (pts - t) @ R
    rgb = cv2.cvtColor(img_bgr,cv2.COLOR_BGR2RGB)
    if (hd,wd)!=(hi,wi): rgb=cv2.resize(rgb,(wd,hd))
    cols = rgb[mask].astype(np.float32)/255.0
    return pts.astype(np.float32), cols

# Map image filenames to paths
name_to_path = {os.path.basename(p):p for p in img_files}
registered   = [n for n in name_to_path if n in IMAGES]
print(f'COLMAP registered {len(registered)}/{len(img_files)} images')

# Sample evenly up to MAX_IMAGES_FOR_DENSE
if len(registered) > MAX_IMAGES_FOR_DENSE:
    idx = np.linspace(0,len(registered)-1,MAX_IMAGES_FOR_DENSE,dtype=int)
    registered = [registered[i] for i in idx]
print(f'Processing {len(registered)} frames for dense reconstruction...')

FRAME_PCDS = []
for i, name in enumerate(registered):
    meta  = IMAGES[name]
    cam   = CAMERAS[meta['camera_id']]
    img   = cv2.imread(name_to_path[name])
    if img is None: continue
    t0    = time.time()
    depth = estimate_depth(img)
    sc    = depth_scale(meta['R'],meta['t'],PTS3D)
    pts,cols = depth_to_pcd(depth,img,cam,meta['R'],meta['t'],sc)
    if len(pts)<100: print(f'  [{i+1}] SKIP — too few points'); continue
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    pcd.colors = o3d.utility.Vector3dVector(cols)
    pcd = pcd.voxel_down_sample(VOXEL_SIZE*0.5)
    FRAME_PCDS.append(pcd)
    print(f'  [{i+1:2d}/{len(registered)}] {name:30s} {len(pcd.points):>8,} pts  '
          f'scale={sc:.3f}  {time.time()-t0:.1f}s')

print(f'\n✅ Generated {len(FRAME_PCDS)} frame point clouds.')


## Cell 9 — ICP Registration
Aligns each frame to the growing scene using **Point-to-Plane ICP**.

This corrects small depth-scale inconsistencies between frames that MiDaS introduces.

> Fitness score close to 1.0 = great alignment. Below 0.3 = warning (frame kept but not corrected).

In [ ]:
def icp_register(frame_pcds):
    if len(frame_pcds)<2:
        return frame_pcds
    def prep(pcd, vox):
        d = pcd.voxel_down_sample(vox)
        d.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=vox*2,max_nn=30))
        d.orient_normals_towards_camera_location(np.zeros(3))
        return d
    vox = VOXEL_SIZE*2
    registered = [frame_pcds[0]]
    target     = copy.deepcopy(frame_pcds[0])
    print(f'Registering {len(frame_pcds)-1} frames to frame 0...')
    for i in range(1,len(frame_pcds)):
        src = frame_pcds[i]
        res = o3d.pipelines.registration.registration_icp(
            prep(src,vox), prep(target,vox),
            max_correspondence_distance=ICP_MAX_CORRESPONDENCE_DISTANCE,
            init=np.eye(4),
            estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
            criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=ICP_MAX_ITERATIONS)
        )
        T = res.transformation if res.fitness>0.05 else np.eye(4)
        status = '✓' if res.fitness>0.3 else '⚠ low fitness'
        print(f'  Frame {i:2d}: fitness={res.fitness:.4f}  RMSE={res.inlier_rmse:.5f}  '
              f'shift={np.linalg.norm(T[:3,3]):.4f}m  {status}')
        aligned = copy.deepcopy(src)
        aligned.transform(T)
        registered.append(aligned)
        target = target + aligned
        if i%5==0:
            target = target.voxel_down_sample(vox*0.5)
    return registered

t0 = time.time()
REG_PCDS = icp_register(FRAME_PCDS)
print(f'\n✅ ICP complete in {time.time()-t0:.1f}s  ({len(REG_PCDS)} frames aligned)')

## Cell 10 — Merge + Clean
Merges all frames into one cloud, then removes noise using:
- **Voxel downsampling** — removes redundant overlapping points
- **Statistical Outlier Removal (SOR)** — removes flying pixel artifacts

In [ ]:
merged = o3d.geometry.PointCloud()
for pcd in REG_PCDS: merged += pcd
n_raw = len(merged.points)
print(f'Merged cloud    : {n_raw:,} points')

merged = merged.voxel_down_sample(VOXEL_SIZE)
n_down = len(merged.points)
print(f'After voxel ({VOXEL_SIZE}m): {n_down:,} ({100*n_down/n_raw:.1f}% retained)')

CLEAN_PCD, _ = merged.remove_statistical_outlier(nb_neighbors=SOR_NB_NEIGHBORS, std_ratio=SOR_STD_RATIO)
n_clean = len(CLEAN_PCD.points)
print(f'After SOR       : {n_clean:,} ({100*n_clean/n_down:.1f}% retained)')

# Visualize in matplotlib (top/front/side views)
pts = np.asarray(CLEAN_PCD.points)
cols = np.asarray(CLEAN_PCD.colors)
step = max(1, len(pts)//5000)
fig, axes = plt.subplots(1,3,figsize=(15,5))
views = [(0,1,'X','Y','Top (XY)'), (0,2,'X','Z','Front (XZ)'), (1,2,'Y','Z','Side (YZ)')]
for ax,(xi,yi,xl,yl,title) in zip(axes,views):
    ax.scatter(pts[::step,xi],pts[::step,yi],c=cols[::step],s=0.3)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(title)
    ax.set_aspect('equal','box')
plt.suptitle(f'Clean Point Cloud — {n_clean:,} points', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER,'clean_pointcloud.png'),dpi=150,bbox_inches='tight')
plt.show()
print(f'✅ Clean point cloud ready.')

## Cell 11 — Tesla-Style Visualization Coloring
Applies the same height-based plasma colormap used in **Tesla's BEV (Bird's Eye View)**
occupancy network visualization:
- Ground level → **purple/blue**
- Mid-height objects → **green/yellow**
- Tall structures → **orange/red**

In [ ]:
def apply_tesla_color(pcd):
    pts = np.asarray(pcd.points)
    n   = len(pts)
    if COLOR_MODE=='height':
        z    = pts[:,2]
        z_lo = np.percentile(z,HEIGHT_CLIP_LOW)
        z_hi = np.percentile(z,HEIGHT_CLIP_HIGH)
        norm = np.clip((z-z_lo)/(z_hi-z_lo+1e-8),0,1)
        colors = plt.get_cmap(COLOR_MAP)(norm)[:,:3]
    elif COLOR_MODE=='rgb':
        if pcd.has_colors(): return pcd
        colors = np.tile([0.65,0.65,0.65],(n,1))
    elif COLOR_MODE=='distance':
        c = pts.mean(axis=0)
        d = np.linalg.norm(pts-c,axis=1)
        norm = (d-d.min())/(d.max()-d.min()+1e-8)
        colors = plt.get_cmap(COLOR_MAP)(norm)[:,:3]
    elif COLOR_MODE=='normals':
        if not pcd.has_normals():
            pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=VOXEL_SIZE*3,max_nn=30))
        colors = np.abs(np.asarray(pcd.normals))
    else:
        colors = np.tile([0.5,0.5,0.5],(n,1))
    pcd.colors = o3d.utility.Vector3dVector(colors.astype(np.float64))
    return pcd

COLORED_PCD = apply_tesla_color(copy.deepcopy(CLEAN_PCD))

# Preview
pts  = np.asarray(COLORED_PCD.points)
cols = np.asarray(COLORED_PCD.colors)
step = max(1,len(pts)//8000)
fig  = plt.figure(figsize=(12,5))
ax1  = fig.add_subplot(121,projection='3d')
ax1.scatter(pts[::step,0],pts[::step,1],pts[::step,2],c=cols[::step],s=0.4)
ax1.set_title(f'Tesla-style {COLOR_MAP} coloring (3D)')
ax2  = fig.add_subplot(122)
ax2.scatter(pts[::step,0],pts[::step,1],c=cols[::step],s=0.3)
ax2.set_title('Bird\'s Eye View (BEV)')
ax2.set_aspect('equal','box')
plt.suptitle(f'Color mode: {COLOR_MODE}  |  Colormap: {COLOR_MAP}',fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER,'tesla_colored_cloud.png'),dpi=150,bbox_inches='tight')
plt.show()
print('✅ Tesla coloring applied.')

## Cell 12 — Poisson Surface Mesh Reconstruction
Converts the colored point cloud into a smooth **watertight triangle mesh**.

> ⏱️ This takes **2–8 minutes** depending on `POISSON_DEPTH`. Go grab a coffee ☕

In [ ]:
# ============================================================
# Cell 12 - Poisson Mesh (No Color Transfer, Tesla-Style Mesh Color)
# ============================================================
import gc, copy
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Optional cleanup of heavy intermediates from earlier cells
for _name in ['merged']:
    if _name in globals():
        del globals()[_name]
gc.collect()

# -- Step 1: Check point cloud size and auto-reduce if needed --
n_pts = len(CLEAN_PCD.points)
print(f"Input point cloud: {n_pts:,} points")

# Conservative limits for Colab RAM safety
SAFE_LIMITS = {
    7: 350_000,
    8: 120_000,
    9: 50_000,
    10: 20_000,
}

safe_depth = POISSON_DEPTH
for depth in sorted(SAFE_LIMITS.keys(), reverse=True):
    if n_pts <= SAFE_LIMITS[depth]:
        safe_depth = depth
        break
else:
    safe_depth = 7

if safe_depth < POISSON_DEPTH:
    print(f"WARNING: POISSON_DEPTH={POISSON_DEPTH} is high for {n_pts:,} points.")
    print(f"         Auto-reduced to depth={safe_depth} for stability.")
else:
    print(f"OK: POISSON_DEPTH={safe_depth} is safe for this point count.")

# -- Step 2: Downsample to safe target for Poisson --
target_max = SAFE_LIMITS[safe_depth]
if n_pts > target_max:
    ratio = target_max / float(n_pts)
    new_voxel = VOXEL_SIZE / (ratio ** (1.0 / 3.0))
    pcd_for_mesh = copy.deepcopy(CLEAN_PCD).voxel_down_sample(new_voxel)
    print(f"Downsampled for Poisson: {n_pts:,} -> {len(pcd_for_mesh.points):,} points (voxel={new_voxel:.4f}m)")
else:
    pcd_for_mesh = copy.deepcopy(CLEAN_PCD)
    print("No extra downsampling needed for Poisson.")

# -- Step 3: Free memory before Poisson --
gc.collect()
try:
    import torch
    if hasattr(device, 'type') and device.type == 'cuda':
        torch.cuda.empty_cache()
        print("GPU cache cleared.")
    else:
        print("CPU mode: no GPU cache to clear.")
except Exception:
    pass

# -- Step 4: Normals (memory-efficient) --
print("\nEstimating normals (memory-efficient settings)...")
pcd_for_mesh.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=VOXEL_SIZE * 3.0,
        max_nn=12,
    )
)
pcd_for_mesh.orient_normals_consistent_tangent_plane(20)
print(f"Normals estimated. Points: {len(pcd_for_mesh.points):,}")

# -- Step 5: Poisson reconstruction --
print(f"\nRunning Poisson reconstruction (depth={safe_depth})...")

try:
    SURFACE_MESH, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd_for_mesh,
        depth=safe_depth,
        linear_fit=False,
    )
    print("Poisson succeeded.")
except Exception as e:
    fallback_depth = max(6, safe_depth - 1)
    print(f"Poisson failed at depth={safe_depth}: {e}")
    print(f"Falling back to depth={fallback_depth}...")
    SURFACE_MESH, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd_for_mesh,
        depth=fallback_depth,
        linear_fit=False,
    )
    safe_depth = fallback_depth
    print(f"Poisson succeeded at depth={safe_depth}.")

# -- Step 6: Remove low-density artifacts --
if MESH_DENSITY_QUANTILE > 0:
    dens = np.asarray(densities, dtype=np.float32)
    threshold = np.quantile(dens, MESH_DENSITY_QUANTILE)
    SURFACE_MESH.remove_vertices_by_mask(dens < threshold)
    SURFACE_MESH.remove_degenerate_triangles()
    SURFACE_MESH.remove_unreferenced_vertices()

del densities
gc.collect()

# -- Step 7: Tesla-style harsh mesh color (no KDTree color transfer) --
print("\nApplying Tesla-style mesh coloring (no color transfer)...")
verts = np.asarray(SURFACE_MESH.vertices, dtype=np.float32)
if len(verts) == 0:
    raise RuntimeError("Mesh has zero vertices after Poisson/density filtering.")

z = verts[:, 2]
z_low = np.percentile(z, 5)
z_high = np.percentile(z, 95)
zn = np.clip((z - z_low) / (z_high - z_low + 1e-8), 0.0, 1.0)

# High-contrast Tesla-like thermal look
mesh_cols = cm.get_cmap('turbo')(zn)[:, :3].astype(np.float32)
SURFACE_MESH.vertex_colors = o3d.utility.Vector3dVector(mesh_cols.astype(np.float64))
SURFACE_MESH.compute_vertex_normals()

del verts, z, zn, mesh_cols
gc.collect()

print("\nMesh complete.")
print(f"Vertices  : {len(SURFACE_MESH.vertices):,}")
print(f"Triangles : {len(SURFACE_MESH.triangles):,}")
print(f"Depth used: {safe_depth}")

# -- Step 8: Lightweight preview --
try:
    vis = o3d.visualization.rendering.OffscreenRenderer(640, 480)
    mat = o3d.visualization.rendering.MaterialRecord()
    mat.shader = "defaultLit"
    vis.scene.add_geometry("mesh", SURFACE_MESH, mat)
    vis.scene.set_background([0.08, 0.08, 0.10, 1.0])
    bounds = SURFACE_MESH.get_axis_aligned_bounding_box()
    vis.setup_camera(55.0, bounds, bounds.get_center())
    img = np.asarray(vis.render_to_image())

    plt.figure(figsize=(8, 5))
    plt.imshow(img)
    plt.title(f"Tesla-style Mesh (depth={safe_depth}) - {len(SURFACE_MESH.triangles):,} triangles", fontweight="bold")
    plt.axis("off")
    plt.savefig(f"{OUTPUT_FOLDER}/mesh_render.png", dpi=140, bbox_inches="tight")
    plt.show()

    vis.scene.clear_geometry()
    del vis
    print("Render saved to output folder.")
except Exception as e:
    print(f"[Note] Offscreen render unavailable ({e}).")
    print("Proceed to export/view in the next cell.")



## Cell 13 — Tesla BEV Voxel Occupancy Mesh
Builds a grid of colored cubes — one cube per occupied voxel — exactly like the
**Tesla FSD occupancy network** visualization you see in Tesla's AI Day presentations.

In [ ]:
if GENERATE_VOXEL_MESH:
    vox_pcd = copy.deepcopy(CLEAN_PCD).voxel_down_sample(VOXEL_MESH_CUBE_SIZE)
    apply_tesla_color(vox_pcd)
    vpts  = np.asarray(vox_pcd.points)
    vcols = np.asarray(vox_pcd.colors)
    n     = len(vpts)
    print(f'Building voxel mesh: {n:,} cubes × 12 triangles = {n*12:,} triangles')

    unit_v = np.array([[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]],dtype=np.float64)-0.5
    unit_t = np.array([[0,2,1],[0,3,2],[4,5,6],[4,6,7],[0,1,5],[0,5,4],[2,3,7],[2,7,6],[0,4,7],[0,7,3],[1,2,6],[1,6,5]],dtype=np.int32)
    all_v  = (unit_v*VOXEL_MESH_CUBE_SIZE + vpts[:,np.newaxis,:]).reshape(-1,3)
    offs   = (np.arange(n)*8)[:,np.newaxis,np.newaxis]
    all_t  = (unit_t[np.newaxis]+offs).reshape(-1,3)
    all_c  = np.repeat(vcols,8,axis=0)

    VOXEL_MESH = o3d.geometry.TriangleMesh()
    VOXEL_MESH.vertices      = o3d.utility.Vector3dVector(all_v)
    VOXEL_MESH.triangles     = o3d.utility.Vector3iVector(all_t)
    VOXEL_MESH.vertex_colors = o3d.utility.Vector3dVector(all_c)
    VOXEL_MESH.compute_vertex_normals()
    print(f'✅ Voxel mesh built: {n:,} cubes')

    # BEV visualization
    step = max(1,n//5000)
    fig,axes = plt.subplots(1,2,figsize=(14,6))
    axes[0].scatter(vpts[::step,0],vpts[::step,1],c=vcols[::step],s=1.5)
    axes[0].set_title('Bird\'s Eye View (BEV)'); axes[0].set_aspect('equal')
    ax3d = fig.add_subplot(122,projection='3d') if False else axes[1]
    axes[1].scatter(vpts[::step,0],vpts[::step,2],c=vcols[::step],s=1.0)
    axes[1].set_title('Front View (XZ)'); axes[1].set_aspect('equal')
    plt.suptitle('Tesla-Style Voxel Occupancy Grid',fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_FOLDER,'voxel_mesh_bev.png'),dpi=150,bbox_inches='tight')
    plt.show()
else:
    VOXEL_MESH = None
    print('Voxel mesh skipped (GENERATE_VOXEL_MESH = False)')

## Cell 14 — Export + Download
Saves all outputs to your Google Drive folder, then auto-downloads the `.glb` file.

> 📁 All files are also saved to `OUTPUT_FOLDER` on Drive so you won't lose them if Colab disconnects.

In [ ]:
from google.colab import files

def export_glb(mesh, path):
    v = np.asarray(mesh.vertices)
    f = np.asarray(mesh.triangles)
    c = np.asarray(mesh.vertex_colors)
    rgba = np.ones((len(v),4),dtype=np.uint8)*255
    rgba[:,:3] = (np.clip(c,0,1)*255).astype(np.uint8)
    tm = trimesh.Trimesh(vertices=v,faces=f,vertex_colors=rgba,process=False)
    tm.fix_normals()
    tm.export(path)
    sz = os.path.getsize(path)/1e6
    print(f'  GLB  → {path}  ({sz:.1f} MB)')
    return path

base = os.path.join(OUTPUT_FOLDER, OUTPUT_NAME)
print('Saving outputs...')

# PLY point cloud
if EXPORT_PLY:
    ply_path = base+'_pointcloud.ply'
    o3d.io.write_point_cloud(ply_path, COLORED_PCD, write_ascii=False)
    print(f'  PLY  → {ply_path}  ({os.path.getsize(ply_path)/1e6:.1f} MB)')

# GLB surface mesh
glb_surface = None
if EXPORT_GLB:
    glb_surface = export_glb(SURFACE_MESH, base+'_surface.glb')
    if VOXEL_MESH is not None:
        export_glb(VOXEL_MESH, base+'_voxel_bev.glb')

print('\n✅ All files saved to Drive.')
print(f'   Folder: {OUTPUT_FOLDER}')
print()

# List all output files
print('Output files:')
for fn in sorted(os.listdir(OUTPUT_FOLDER)):
    fp = os.path.join(OUTPUT_FOLDER,fn)
    if os.path.isfile(fp):
        print(f'  {fn:45s} {os.path.getsize(fp)/1e6:.1f} MB')

# Auto-download the GLB file to your computer
if glb_surface and os.path.exists(glb_surface):
    print(f'\n⬇️  Downloading {os.path.basename(glb_surface)} to your computer...')
    files.download(glb_surface)

print('\n🎉 Done! View your .GLB at: https://gltf-viewer.donmccurdy.com')
print('   Drag and drop the downloaded .glb file into that page.')